# 05 베이스라인 비교 및 메타데이터 효과 검증

04번에서 학습한 제안 모델 (Hybrid MLP: KcBERT PCA + Metadata)이 실제로 좋은 모델인지 확인하기 위해 베이스라인과 비교한다.

이 노트북의 목적은 다음과 같다.

1. `TF-IDF + Random Forest` 베이스라인을 학습한다.
2. `KcBERT PCA only + MLP`, 비교 모델 (Metadata only + MLP)을 학습한다.
3. 04번의 제안 모델 (Hybrid MLP: KcBERT PCA + Metadata) 결과와 같은 지표로 비교한다.
4. 메타데이터 결합이 F1, PR-AUC, Recall을 실제로 개선했는지 확인한다.
5. 최종 모델 선택과 오답 분석을 위한 06번 입력 자료를 만든다.

05번은 아직 별점 정제를 수행하지 않는다. 별점 정제는 06번에서 최종 모델을 선택한 뒤 07번에서 진행한다.


## 1. 라이브러리 로드

- 04번과 같은 평가 지표를 사용한다.
- MLP 계열 비교 모델은 train 데이터에만 SMOTE를 적용한다.
- TF-IDF + Random Forest는 sparse text feature를 사용하므로 SMOTE를 적용하지 않고 `class_weight='balanced'`로 불균형을 보정한다.


In [1]:
import json
import os

import joblib
import numpy as np
import pandas as pd
from IPython.display import display
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

os.makedirs('outputs', exist_ok=True)


## 2. 데이터 로드 및 split 재현

- `final_hybrid_train/val/test.csv`는 03번에서 만든 PCA + metadata 결합 feature이다.
- TF-IDF 베이스라인은 원문 텍스트가 필요하므로 `preprocessed_reviews.csv`를 다시 불러온다.
- 03번과 같은 `random_state=42`, `stratify=label` 조건으로 train/validation/test index를 재현한다.
- 재현한 index의 label 순서가 03번 산출물의 `target_label`과 같은지 확인한다.


In [2]:
SEED = 42
TARGET_COL = 'target_label'
LABEL_COL = 'label'
TEXT_COL = 'cleaned_review_text'
META_COLS = ['meta_rating', 'meta_text_length', 'meta_emoji_count', 'meta_photo_count']
MIN_TUNED_THRESHOLD = 0.5

train_df = pd.read_csv('csv/final_hybrid_train.csv')
val_df = pd.read_csv('csv/final_hybrid_val.csv')
test_df = pd.read_csv('csv/final_hybrid_test.csv')
raw_df = pd.read_csv('csv/preprocessed_reviews.csv')

pca_cols = [col for col in train_df.columns if col.startswith('feat_')]
meta_cols = META_COLS
hybrid_cols = pca_cols + meta_cols

train_idx, temp_idx = train_test_split(
    raw_df.index,
    test_size=0.3,
    random_state=SEED,
    stratify=raw_df[LABEL_COL],
)
val_idx, test_idx = train_test_split(
    temp_idx,
    test_size=0.5,
    random_state=SEED,
    stratify=raw_df.loc[temp_idx, LABEL_COL],
)

text_train = raw_df.loc[train_idx, TEXT_COL].fillna('').astype(str)
text_val = raw_df.loc[val_idx, TEXT_COL].fillna('').astype(str)
text_test = raw_df.loc[test_idx, TEXT_COL].fillna('').astype(str)

y_train = train_df[TARGET_COL].astype(int)
y_val = val_df[TARGET_COL].astype(int)
y_test = test_df[TARGET_COL].astype(int)

np.testing.assert_array_equal(raw_df.loc[train_idx, LABEL_COL].astype(int).to_numpy(), y_train.to_numpy())
np.testing.assert_array_equal(raw_df.loc[val_idx, LABEL_COL].astype(int).to_numpy(), y_val.to_numpy())
np.testing.assert_array_equal(raw_df.loc[test_idx, LABEL_COL].astype(int).to_numpy(), y_test.to_numpy())

print('Hybrid train:', train_df.shape)
print('Hybrid validation:', val_df.shape)
print('Hybrid test:', test_df.shape)
print('PCA feature 수:', len(pca_cols))
print('metadata feature:', meta_cols)
print('train label 분포:', y_train.value_counts().sort_index().to_dict())
print('validation label 분포:', y_val.value_counts().sort_index().to_dict())
print('test label 분포:', y_test.value_counts().sort_index().to_dict())


Hybrid train: (6188, 148)
Hybrid validation: (1326, 148)
Hybrid test: (1326, 148)
PCA feature 수: 143
metadata feature: ['meta_rating', 'meta_text_length', 'meta_emoji_count', 'meta_photo_count']
train label 분포: {0: 3983, 1: 2205}
validation label 분포: {0: 854, 1: 472}
test label 분포: {0: 853, 1: 473}


## 3. 공통 평가 함수 정의

04코드와 같은 방식으로 평가한다.

- 기본 threshold는 `0.5`이다.
- 추가로 validation 데이터에서 F1이 가장 높은 threshold를 찾는다.
- 단, threshold가 지나치게 낮아져 일반 리뷰를 과도하게 이벤트 리뷰로 오탐하지 않도록 `0.5 이상` 후보만 사용한다.


In [3]:
def tune_threshold_from_validation(y_true, prob, min_threshold=0.5):
    precisions, recalls, thresholds = precision_recall_curve(y_true, prob)

    if len(thresholds) == 0:
        return float(min_threshold), pd.DataFrame()

    f1s = 2 * precisions[:-1] * recalls[:-1] / (precisions[:-1] + recalls[:-1] + 1e-8)
    candidates = pd.DataFrame({
        'threshold': thresholds,
        'precision': precisions[:-1],
        'recall': recalls[:-1],
        'f1': f1s,
    })
    candidates = candidates[candidates['threshold'] >= min_threshold]

    if candidates.empty:
        return float(min_threshold), candidates

    best_threshold = float(candidates.sort_values('f1', ascending=False).iloc[0]['threshold'])
    return best_threshold, candidates.sort_values('f1', ascending=False).reset_index(drop=True)


def metric_row(model_name, feature_set, split, y_true, prob, threshold):
    pred = (prob >= threshold).astype(int)
    cm = confusion_matrix(y_true, pred, labels=[0, 1])
    return {
        'model': model_name,
        'feature_set': feature_set,
        'split': split,
        'threshold': float(threshold),
        'f1': float(f1_score(y_true, pred)),
        'pr_auc': float(average_precision_score(y_true, prob)),
        'precision': float(precision_score(y_true, pred, zero_division=0)),
        'recall': float(recall_score(y_true, pred, zero_division=0)),
        'roc_auc': float(roc_auc_score(y_true, prob)),
        'tn': int(cm[0, 0]),
        'fp': int(cm[0, 1]),
        'fn': int(cm[1, 0]),
        'tp': int(cm[1, 1]),
    }


def evaluate_probabilities(model_name, feature_set, y_val, val_prob, y_test, test_prob):
    best_threshold, threshold_candidates = tune_threshold_from_validation(
        y_val,
        val_prob,
        min_threshold=MIN_TUNED_THRESHOLD,
    )

    rows = [
        metric_row(model_name, feature_set, 'validation_default_0_5', y_val, val_prob, 0.5),
        metric_row(model_name, feature_set, 'validation_tuned_min_0_5', y_val, val_prob, best_threshold),
        metric_row(model_name, feature_set, 'test_default_0_5', y_test, test_prob, 0.5),
        metric_row(model_name, feature_set, 'test_tuned_min_0_5', y_test, test_prob, best_threshold),
    ]
    return rows, best_threshold, threshold_candidates


## 4. 04번 제안 모델 (Hybrid MLP: KcBERT PCA + Metadata) 결과 로드

04번에서 이미 학습한 `KcBERT PCA + Metadata Hybrid + MLP` 결과를 불러온다.

이 결과가 05번 비교표에서 제안 모델 (Hybrid MLP: KcBERT PCA + Metadata) 역할을 한다.


In [4]:
proposed_metrics = pd.read_csv('outputs/proposed_mlp_metrics.csv')
proposed_metrics.insert(0, 'feature_set', 'kcbert_pca+metadata')
proposed_metrics.insert(0, 'model', 'proposed_hybrid_mlp_04')

with open('outputs/proposed_mlp_selected_config.json', 'r', encoding='utf-8') as f:
    proposed_config = json.load(f)

proposed_metrics


,model,feature_set,split,threshold,f1,pr_auc,precision,recall,roc_auc,tn,fp,fn,tp
0,proposed_hybrid_mlp_04,kcbert_pca+metadata,validation_default_0_5,0.500000,0.425656,0.395151,0.393178,0.463983,0.553857,516,338,253,219
1,proposed_hybrid_mlp_04,kcbert_pca+metadata,validation_tuned_min_0_5,0.507434,0.426197,0.395151,0.395644,0.461864,0.553857,521,333,254,218
2,proposed_hybrid_mlp_04,kcbert_pca+metadata,test_default_0_5,0.500000,0.428975,0.380842,0.386441,0.482030,0.537444,491,362,245,228
3,proposed_hybrid_mlp_04,kcbert_pca+metadata,test_tuned_min_0_5,0.507434,0.427350,0.380842,0.387931,0.475687,0.537444,498,355,248,225


## 5. TF-IDF + Random Forest 베이스라인

텍스트만 사용하는 전통적인 베이스라인이다.

- 입력: `cleaned_review_text`
- feature: TF-IDF unigram/bigram
- classifier: RandomForestClassifier
- 불균형 보정: `class_weight='balanced'`

이 모델과 제안 모델 (Hybrid MLP: KcBERT PCA + Metadata)을 비교하면, KcBERT 임베딩과 메타데이터 결합 전략이 단순 텍스트 베이스라인보다 나은지 확인할 수 있다.


In [5]:
tfidf_rf = Pipeline([
    ('tfidf', TfidfVectorizer(
        max_features=20000,
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.95,
    )),
    ('rf', RandomForestClassifier(
        n_estimators=300,
        random_state=SEED,
        n_jobs=-1,
        class_weight='balanced',
    )),
])

tfidf_rf.fit(text_train, y_train)

tfidf_val_prob = tfidf_rf.predict_proba(text_val)[:, 1]
tfidf_test_prob = tfidf_rf.predict_proba(text_test)[:, 1]

tfidf_rows, tfidf_best_threshold, tfidf_threshold_candidates = evaluate_probabilities(
    'baseline_tfidf_random_forest',
    'cleaned_text_tfidf',
    y_val,
    tfidf_val_prob,
    y_test,
    tfidf_test_prob,
)

joblib.dump({
    'model': tfidf_rf,
    'model_name': 'baseline_tfidf_random_forest',
    'text_col': TEXT_COL,
    'target_col': LABEL_COL,
    'default_threshold': 0.5,
    'best_threshold': tfidf_best_threshold,
}, 'outputs/baseline_tfidf_random_forest_model.joblib')

print('TF-IDF + Random Forest best threshold:', tfidf_best_threshold)
display(pd.DataFrame(tfidf_rows).round(4))


TF-IDF + Random Forest best threshold: 0.5021455505279036


,model,feature_set,split,threshold,f1,pr_auc,precision,recall,roc_auc,tn,fp,fn,tp
0,baseline_tfidf_random_forest,cleaned_text_tfidf,validation_default_0_5,0.5000,0.2438,0.3782,0.3971,0.1758,0.5343,728,126,389,83
1,baseline_tfidf_random_forest,cleaned_text_tfidf,validation_tuned_min_0_5,0.5021,0.2459,0.3782,0.4089,0.1758,0.5343,734,120,389,83
2,baseline_tfidf_random_forest,cleaned_text_tfidf,test_default_0_5,0.5000,0.2511,0.3956,0.4167,0.1797,0.5356,734,119,388,85
3,baseline_tfidf_random_forest,cleaned_text_tfidf,test_tuned_min_0_5,0.5021,0.2496,0.3956,0.4200,0.1776,0.5356,737,116,389,84


## 6. KcBERT PCA only / 비교 모델 (Metadata only + MLP) 비교

04번의 제안 모델 (Hybrid MLP: KcBERT PCA + Metadata)이 좋은 이유가 텍스트 임베딩 때문인지, 메타데이터 때문인지, 둘의 결합 때문인지 확인하기 위한 절제 비교다.

- `KcBERT PCA only + MLP`: PCA feature만 사용
- 비교 모델 (Metadata only + MLP): metadata feature만 사용
- `KcBERT PCA + Metadata Hybrid + MLP`: 04번 결과 사용

MLP 구조는 04번에서 선택된 `mlp_32` 설정을 그대로 사용한다.


In [6]:
def make_mlp_model(random_state=SEED):
    return Pipeline([
        ('scaler', StandardScaler()),
        ('mlp', MLPClassifier(
            hidden_layer_sizes=tuple(proposed_config['hidden_layer_sizes']),
            activation='relu',
            alpha=proposed_config['alpha'],
            batch_size=64,
            learning_rate_init=1e-3,
            max_iter=200,
            early_stopping=True,
            validation_fraction=0.15,
            n_iter_no_change=10,
            random_state=random_state,
        )),
    ])


def fit_mlp_variant(model_name, feature_set, cols):
    X_train = train_df[cols]
    X_val = val_df[cols]
    X_test = test_df[cols]

    X_smote, y_smote = SMOTE(random_state=SEED).fit_resample(X_train, y_train)

    model = make_mlp_model(random_state=SEED)
    model.fit(X_smote, y_smote)

    val_prob = model.predict_proba(X_val)[:, 1]
    test_prob = model.predict_proba(X_test)[:, 1]

    rows, best_threshold, threshold_candidates = evaluate_probabilities(
        model_name,
        feature_set,
        y_val,
        val_prob,
        y_test,
        test_prob,
    )
    return model, rows, best_threshold, threshold_candidates


pca_only_model, pca_only_rows, pca_only_threshold, pca_only_candidates = fit_mlp_variant(
    'ablation_mlp_kcbert_pca_only',
    'kcbert_pca_only',
    pca_cols,
)

metadata_only_model, metadata_only_rows, metadata_only_threshold, metadata_only_candidates = fit_mlp_variant(
    'ablation_mlp_metadata_only',
    'metadata_only',
    meta_cols,
)

joblib.dump({
    'model': pca_only_model,
    'model_name': 'ablation_mlp_kcbert_pca_only',
    'feature_cols': pca_cols,
    'target_col': TARGET_COL,
    'default_threshold': 0.5,
    'best_threshold': pca_only_threshold,
}, 'outputs/ablation_mlp_kcbert_pca_only_model.joblib')

joblib.dump({
    'model': metadata_only_model,
    'model_name': 'ablation_mlp_metadata_only',
    'feature_cols': meta_cols,
    'target_col': TARGET_COL,
    'default_threshold': 0.5,
    'best_threshold': metadata_only_threshold,
}, 'outputs/ablation_mlp_metadata_only_model.joblib')

print('KcBERT PCA only best threshold:', pca_only_threshold)
print('Metadata only best threshold:', metadata_only_threshold)
display(pd.DataFrame(pca_only_rows + metadata_only_rows).round(4))


KcBERT PCA only best threshold: 0.503822902158696
Metadata only best threshold: 0.5003768331844837


,model,feature_set,split,threshold,f1,pr_auc,precision,recall,roc_auc,tn,fp,fn,tp
0,ablation_mlp_kcbert_pca_only,kcbert_pca_only,validation_default_0_5,0.5000,0.4268,0.4197,0.4157,0.4386,0.5763,563,291,265,207
1,ablation_mlp_kcbert_pca_only,kcbert_pca_only,validation_tuned_min_0_5,0.5038,0.4286,0.4197,0.4190,0.4386,0.5763,567,287,265,207
2,ablation_mlp_kcbert_pca_only,kcbert_pca_only,test_default_0_5,0.5000,0.3904,0.3754,0.3691,0.4144,0.5291,518,335,277,196
3,ablation_mlp_kcbert_pca_only,kcbert_pca_only,test_tuned_min_0_5,0.5038,0.3908,0.3754,0.3714,0.4123,0.5291,523,330,278,195
4,ablation_mlp_metadata_only,metadata_only,validation_default_0_5,0.5000,0.5209,0.3987,0.4067,0.7246,0.5791,355,499,130,342
5,ablation_mlp_metadata_only,metadata_only,validation_tuned_min_0_5,0.5004,0.5209,0.3987,0.4067,0.7246,0.5791,355,499,130,342
6,ablation_mlp_metadata_only,metadata_only,test_default_0_5,0.5000,0.5270,0.4363,0.4111,0.7336,0.6031,356,497,126,347
7,ablation_mlp_metadata_only,metadata_only,test_tuned_min_0_5,0.5004,0.5258,0.4363,0.4104,0.7315,0.6031,356,497,127,346


## 7. 메타데이터 영향도 분석

04번에서 저장한 제안 모델 (Hybrid MLP: KcBERT PCA + Metadata)을 불러와 validation 데이터 기준 permutation importance를 계산한다.

각 메타데이터 컬럼을 하나씩 섞었을 때 F1이 얼마나 떨어지는지 확인한다. F1 감소폭이 클수록 해당 메타데이터가 모델 예측에 더 큰 영향을 준 것으로 해석한다.


In [7]:
def metadata_permutation_importance(model, X_val, y_val, metadata_cols, threshold, repeats=10, random_state=SEED):
    rng = np.random.default_rng(random_state)
    base_prob = model.predict_proba(X_val)[:, 1]
    base_pred = (base_prob >= threshold).astype(int)
    base_f1 = f1_score(y_val, base_pred)

    rows = []
    for col in metadata_cols:
        drops = []
        for _ in range(repeats):
            X_perm = X_val.copy()
            shuffled = X_perm[col].to_numpy().copy()
            rng.shuffle(shuffled)
            X_perm[col] = shuffled

            perm_prob = model.predict_proba(X_perm)[:, 1]
            perm_pred = (perm_prob >= threshold).astype(int)
            perm_f1 = f1_score(y_val, perm_pred)
            drops.append(base_f1 - perm_f1)

        rows.append({
            'metadata': col,
            'base_f1': float(base_f1),
            'mean_f1_drop': float(np.mean(drops)),
            'std_f1_drop': float(np.std(drops)),
            'repeats': repeats,
        })

    return pd.DataFrame(rows).sort_values('mean_f1_drop', ascending=False).reset_index(drop=True)


proposed_bundle = joblib.load('outputs/proposed_mlp_final_model.joblib')
proposed_model = proposed_bundle['model']
proposed_feature_cols = proposed_bundle['feature_cols']
proposed_threshold = proposed_bundle.get('best_threshold', proposed_config['best_threshold_from_validation'])

metadata_importance = metadata_permutation_importance(
    proposed_model,
    val_df[proposed_feature_cols],
    y_val,
    meta_cols,
    proposed_threshold,
    repeats=10,
)

metadata_importance_display = metadata_importance.copy()
metadata_importance_display.insert(0, 'rank', metadata_importance_display.index + 1)
metadata_importance_display = metadata_importance_display.rename(columns={
    'metadata': '메타데이터',
    'base_f1': '기준 F1',
    'mean_f1_drop': '평균 F1 감소폭',
    'std_f1_drop': 'F1 감소폭 표준편차',
    'repeats': '반복 횟수',
})
metadata_importance_display = metadata_importance_display.round({
    '기준 F1': 4,
    '평균 F1 감소폭': 4,
    'F1 감소폭 표준편차': 4,
})

top_metadata = metadata_importance.iloc[0]
print(
    f"가장 큰 영향을 준 메타데이터: {top_metadata['metadata']} "
    f"(평균 F1 감소폭: {top_metadata['mean_f1_drop']:.4f})"
)
display(metadata_importance_display)


가장 큰 영향을 준 메타데이터: meta_rating (평균 F1 감소폭: 0.0128)


,rank,메타데이터,기준 F1,평균 F1 감소폭,F1 감소폭 표준편차,반복 횟수
0,1,meta_rating,0.4262,0.0128,0.0026,10
1,2,meta_photo_count,0.4262,0.0108,0.0067,10
2,3,meta_text_length,0.4262,0.0081,0.0043,10
3,4,meta_emoji_count,0.4262,-0.0018,0.0026,10


## 8. 전체 비교표 출력 및 모델 저장

04번 제안 모델 (Hybrid MLP: KcBERT PCA + Metadata) 결과와 05번에서 학습한 baseline/ablation 결과를 하나의 표로 합친다.

성능 비교 결과는 노트북 셀에 바로 출력하고, 모델 객체만 `.joblib` 파일로 따로 저장한다. 최종 비교는 `test_tuned_min_0_5` 기준을 우선 확인한다.


In [8]:
baseline_metrics = pd.concat([
    proposed_metrics,
    pd.DataFrame(tfidf_rows),
    pd.DataFrame(pca_only_rows),
    pd.DataFrame(metadata_only_rows),
], ignore_index=True)

comparison_summary = (
    baseline_metrics[baseline_metrics['split'] == 'test_tuned_min_0_5']
    .sort_values(['f1', 'pr_auc'], ascending=False)
    .reset_index(drop=True)
)
comparison_summary.insert(0, 'rank', comparison_summary.index + 1)

comparison_summary_display = comparison_summary.round({
    'threshold': 4,
    'f1': 4,
    'pr_auc': 4,
    'precision': 4,
    'recall': 4,
    'roc_auc': 4,
})

baseline_metrics_display = (
    baseline_metrics
    .sort_values(['model', 'split'])
    .reset_index(drop=True)
    .round({
        'threshold': 4,
        'f1': 4,
        'pr_auc': 4,
        'precision': 4,
        'recall': 4,
        'roc_auc': 4,
    })
)

saved_model_files = pd.DataFrame([
    {'model': 'baseline_tfidf_random_forest', 'path': 'outputs/baseline_tfidf_random_forest_model.joblib'},
    {'model': 'ablation_mlp_kcbert_pca_only', 'path': 'outputs/ablation_mlp_kcbert_pca_only_model.joblib'},
    {'model': 'ablation_mlp_metadata_only', 'path': 'outputs/ablation_mlp_metadata_only_model.joblib'},
    {'model': 'proposed_hybrid_mlp_04', 'path': 'outputs/proposed_mlp_final_model.joblib'},
])

print('Test tuned 기준 전체 모델 순위')
display(comparison_summary_display)

print('Validation/Test 전체 평가 결과')
display(baseline_metrics_display)

print('저장된 모델 파일')
display(saved_model_files)


Test tuned 기준 전체 모델 순위


,rank,model,feature_set,split,threshold,f1,pr_auc,precision,recall,roc_auc,tn,fp,fn,tp
0,1,ablation_mlp_metadata_only,metadata_only,test_tuned_min_0_5,0.5004,0.5258,0.4363,0.4104,0.7315,0.6031,356,497,127,346
1,2,proposed_hybrid_mlp_04,kcbert_pca+metadata,test_tuned_min_0_5,0.5074,0.4274,0.3808,0.3879,0.4757,0.5374,498,355,248,225
2,3,ablation_mlp_kcbert_pca_only,kcbert_pca_only,test_tuned_min_0_5,0.5038,0.3908,0.3754,0.3714,0.4123,0.5291,523,330,278,195
3,4,baseline_tfidf_random_forest,cleaned_text_tfidf,test_tuned_min_0_5,0.5021,0.2496,0.3956,0.4200,0.1776,0.5356,737,116,389,84


Validation/Test 전체 평가 결과


,model,feature_set,split,threshold,f1,pr_auc,precision,recall,roc_auc,tn,fp,fn,tp
0,ablation_mlp_kcbert_pca_only,kcbert_pca_only,test_default_0_5,0.5000,0.3904,0.3754,0.3691,0.4144,0.5291,518,335,277,196
1,ablation_mlp_kcbert_pca_only,kcbert_pca_only,test_tuned_min_0_5,0.5038,0.3908,0.3754,0.3714,0.4123,0.5291,523,330,278,195
2,ablation_mlp_kcbert_pca_only,kcbert_pca_only,validation_default_0_5,0.5000,0.4268,0.4197,0.4157,0.4386,0.5763,563,291,265,207
3,ablation_mlp_kcbert_pca_only,kcbert_pca_only,validation_tuned_min_0_5,0.5038,0.4286,0.4197,0.4190,0.4386,0.5763,567,287,265,207
4,ablation_mlp_metadata_only,metadata_only,test_default_0_5,0.5000,0.5270,0.4363,0.4111,0.7336,0.6031,356,497,126,347
5,ablation_mlp_metadata_only,metadata_only,test_tuned_min_0_5,0.5004,0.5258,0.4363,0.4104,0.7315,0.6031,356,497,127,346
6,ablation_mlp_metadata_only,metadata_only,validation_default_0_5,0.5000,0.5209,0.3987,0.4067,0.7246,0.5791,355,499,130,342
7,ablation_mlp_metadata_only,metadata_only,validation_tuned_min_0_5,0.5004,0.5209,0.3987,0.4067,0.7246,0.5791,355,499,130,342
8,baseline_tfidf_random_forest,cleaned_text_tfidf,test_default_0_5,0.5000,0.2511,0.3956,0.4167,0.1797,0.5356,734,119,388,85
9,baseline_tfidf_random_forest,cleaned_text_tfidf,test_tuned_min_0_5,0.5021,0.2496,0.3956,0.4200,0.1776,0.5356,737,116,389,84


저장된 모델 파일


,model,path
0,baseline_tfidf_random_forest,outputs/baseline_tfidf_random_forest_model.joblib
1,ablation_mlp_kcbert_pca_only,outputs/ablation_mlp_kcbert_pca_only_model.joblib
2,ablation_mlp_metadata_only,outputs/ablation_mlp_metadata_only_model.joblib
3,proposed_hybrid_mlp_04,outputs/proposed_mlp_final_model.joblib


## 9. 05번 결과 해석 기준

05번 실행 후에는 다음을 확인한다.

1. 제안 모델 (Hybrid MLP: KcBERT PCA + Metadata, `proposed_hybrid_mlp_04`)이 `baseline_tfidf_random_forest`보다 좋은가?
2. 제안 모델 (Hybrid MLP: KcBERT PCA + Metadata, `proposed_hybrid_mlp_04`)이 `ablation_mlp_kcbert_pca_only`보다 좋은가?
3. 제안 모델 (Hybrid MLP: KcBERT PCA + Metadata, `proposed_hybrid_mlp_04`)이 비교 모델 (Metadata only + MLP, `ablation_mlp_metadata_only`)보다 좋은가?
4. metadata를 결합했을 때 F1, PR-AUC, Recall이 실제로 올라갔는가?
5. 메타데이터 영향도 출력표에서 어떤 메타데이터가 예측에 가장 큰 영향을 줬는가?

이 결과가 06번의 입력이 된다.

06번에서는 다음 작업을 진행한다.

- 최종 모델 선택
- FP/FN 오답 리뷰 추출
- 은어, 신조어, 우회 표현에 대한 실패 사례 분석
- 메타데이터가 텍스트 기반 모호성을 줄였는지 해석

따라서 05번은 별점 정제 단계가 아니라, 제안 모델 (Hybrid MLP: KcBERT PCA + Metadata)의 타당성을 검증하는 단계이다.
